In [4]:
import os
from dotenv import load_dotenv
os.environ["GROQ_API_LKEY"] = os.getenv("GROQ_API_KEY")


### Middleware

Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:

* Tracking agent behavior with logging, analytics, and debugging.
* Transforming prompts, tool selection, and output formatting.
* Adding retries, fallbacks, and early termination logic.
* Applying rate limits, guardrails, and PII detection.

### Summarize MiddleWare

Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:

* Long-running conversations that exceed context windows.
* Multi-turn dialogues with extensive history.
* Applications where preserving full conversation context matters.


In [5]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage
from langchain.chat_models import init_chat_model

# Message based Summarization
llm = init_chat_model(
    model = "openai/gpt-oss-20b",
    model_provider= "groq"
)

agent = create_agent(
    model= llm,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=llm,
            trigger=("messages", 10),
            keep=("messages", 4),
        ),
    ]
)

In [6]:
# Run with thread ID
config = {"configurable": {"thread_id":"test-1"}}

In [7]:
from langchain_core.messages import content
# Alternative test data
questions = [
    "what is 2+2?",
    "what is 10*5?",
    "what is 100/4?",
    "what is 15-7?",
    "what is 3*3?",
    "what is 4*4?",
]

for q in questions:
    response = agent.invoke({"messages":[HumanMessage(content=q)]}, config)

    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='what is 2+2?', additional_kwargs={}, response_metadata={}, id='ac73fdc2-c86d-4c5f-988a-8289c6b32e9b'), AIMessage(content='2\u202f+\u202f2\u202f=\u202f4', additional_kwargs={'reasoning_content': 'The user asks a simple math question: "what is 2+2?" The answer is 4. Provide answer.'}, response_metadata={'token_usage': {'completion_tokens': 44, 'prompt_tokens': 78, 'total_tokens': 122, 'completion_time': 0.047454459, 'completion_tokens_details': {'reasoning_tokens': 26}, 'prompt_time': 0.003711177, 'prompt_tokens_details': None, 'queue_time': 0.335914984, 'total_time': 0.051165636}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_9340e7d14d', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0160c-fdad-7072-97a9-89af43ab3d44-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 78, 'output_tokens': 44, 'total_tokens': 122, 'output_token_details': 

### Token Based Trigger

In [8]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool

@tool
def search_hotels(city: str) -> str:
    """Search hotles - returns long responses to use more tokens"""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""

llm = init_chat_model(
    model= "openai/gpt-oss-20b",
    model_provider= "groq"
)

agent = create_agent(
    model=llm,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
        model= llm,
        trigger=("tokens", 550),
        keep=("tokens", 200),
        ),
    ]
)

config = {"configurable": {"thread_id":"test-1"}}

def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4 # 4 chars = 1 token
    

In [9]:
cities = ["Paris", "London", "New York", "Tokyo", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke({"messages": [HumanMessage(content=f"Find hotels in {city}")]},
    config = config
    )

    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris: ~236 tokens, 2 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='b019f70c-7802-409a-aca3-060ba41c1102'), AIMessage(content='Sure thing! Paris has a wide range of hotels to suit almost any taste and budget. To give you the best options, could you let me know:\n\n1. **Travel dates** (check‑in and check‑out)  \n2. **Budget range** (e.g., €80–€150 per night, €200+ for luxury, etc.)  \n3. **Preferred star rating** (3★, 4★, 5★, boutique, etc.)  \n4. **Key location preferences** – near a particular arrondissement, landmark, or public‑transport hub?  \n5. **Special requirements** – Wi‑Fi only, breakfast included, pet‑friendly, accessible rooms, etc.  \n\nOnce I have those details, I’ll compile a short list of hotels that match your criteria, along with a quick snapshot of each (price range, amenities, and why it might be a good fit). You can then check availability and book directly through the hotel’s website, a travel aggregator (li

### Based on Fraction

In [10]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels."""
    return f"Hotels in {city}: Grand Hotel $350, City Inn $180, Budget Stay $75"

# LOW fraction for testing!
llm = init_chat_model(
    model = "openai/gpt-oss-20b",
    model_provider= "groq"
)


agent = create_agent(
    model=llm,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=llm,
            trigger=("fraction", 0.005), # 0.5% = ~640 tokens
            keep=("fraction", 0.002),    # 0.2% = ~256 tokens
        )
    ]
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter
def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages) // 4

# Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Hotels in {city}")]},
        config=config
    )
    tokens = count_tokens(response["messages"])
    fraction = tokens / 128000  # gpt-4o-mini context
    print(f"{city}: {tokens} tokens ({fraction:.4%}), {len(response['messages'])} msgs")
    print(response['messages'])


Paris: 301 tokens (0.2352%), 4 msgs
[HumanMessage(content='Hotels in Paris', additional_kwargs={}, response_metadata={}, id='9eee671b-1eee-478d-b3e6-08578951f575'), AIMessage(content='', additional_kwargs={'reasoning_content': 'The user says "Hotels in Paris". Likely they want a search for hotels. We should use the search_hotels function with city="Paris".', 'tool_calls': [{'id': 'fc_88cba1cf-06d9-48e4-8272-40cf903cd050', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 55, 'prompt_tokens': 120, 'total_tokens': 175, 'completion_time': 0.066502345, 'completion_tokens_details': {'reasoning_tokens': 31}, 'prompt_time': 0.006603467, 'prompt_tokens_details': None, 'queue_time': 0.26566937, 'total_time': 0.073105812}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_84bb35977d', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq

### Human In the Loop MiddleWare

Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Human-in-the-loop is useful for the following:

* High-stakes operations requiring human approval (e.g. database writes, financial transactions).
* Compliance workflows where human oversight is mandatory.
* Long-running conversations where human feedback guides the agent.


In [1]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to the {recipient} with subject '{subject}'"

In [11]:
llm = init_chat_model(
    model = "openai/gpt-oss-20b",
    model_provider= "groq"
)

agent = create_agent(
    model = llm,
    tools= [read_email_tool, send_email_tool],
    checkpointer= InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve", "edit", "reject"]
                },
                "read_email_tools":False
            }
        )

    ]
)

In [12]:
config = {"configurable": {"thread_id":"test-approve"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send an email to John@test.com with subject 'Hello' and body 'How are you?'")]},
    config = config
)

result

{'messages': [HumanMessage(content="Send an email to John@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='a54ab6a2-af46-4803-8941-99df9a53df96'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call send_email_tool.', 'tool_calls': [{'id': 'fc_11a6977e-1a9b-4854-8ebe-642099d7f8a8', 'function': {'arguments': '{"body":"How are you?","recipient":"John@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 46, 'prompt_tokens': 175, 'total_tokens': 221, 'completion_time': 0.053541977, 'completion_tokens_details': {'reasoning_tokens': 9}, 'prompt_time': 0.008567804, 'prompt_tokens_details': None, 'queue_time': 0.210462882, 'total_time': 0.062109781}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_99996fee8e', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, 

In [13]:
from langgraph.types import Command
# Step 2: Approve
if "__interrupt__" in result:
    print("Paused Approving...")

    result = agent.invoke(
        Command(
            resume={
                "decisions" : [
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )

    print(f"Result: {result['messages'][-1].content}")

Paused Approving...
Result: ✅ Email sent to **John@test.com** with subject **“Hello”** and body **“How are you?”**. Let me know if you need anything else!


In [14]:
result

{'messages': [HumanMessage(content="Send an email to John@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='a54ab6a2-af46-4803-8941-99df9a53df96'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call send_email_tool.', 'tool_calls': [{'id': 'fc_11a6977e-1a9b-4854-8ebe-642099d7f8a8', 'function': {'arguments': '{"body":"How are you?","recipient":"John@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 46, 'prompt_tokens': 175, 'total_tokens': 221, 'completion_time': 0.053541977, 'completion_tokens_details': {'reasoning_tokens': 9}, 'prompt_time': 0.008567804, 'prompt_tokens_details': None, 'queue_time': 0.210462882, 'total_time': 0.062109781}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_99996fee8e', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, 